In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from Base_Modules.Environments import Prison
from Prison_Strategies.Basic_Strategies import *
from Base_Modules.Game_Master import Game_Master
from Base_Modules.Action import Action_History, Prison_Actions, Duel_Matrix
import pandas as pd
from collections import defaultdict

In [3]:
prison = Prison()
actions = prison.Get_Actions()

In [4]:
strategies_list = [
    Random_Strategy(actions=actions),
    Random_Strategy(actions=actions, p_coop=0.1),
    Always_Betray(actions=actions),
    Always_Cooperate(actions=actions),
    Always_Cooperate(actions=actions),
    Patient_Unforgiving(actions=actions),
    Copycat(actions=actions),
    Periodic(actions=actions, period=1),
]

number_of_strategies = len(strategies_list)

In [5]:
def Get_Index_By_Name(strategies : dict[int, Strategy], name : str) -> int:
    for ix, (id, s) in enumerate(strategies.items()):
        if str(s) == name:
            return id
    for ix, (id, s) in enumerate(strategies.items()):
        if str(s).startswith(name):
            return id
    return -1

In [6]:
strategies = {}

for (i, s) in enumerate(strategies_list):
    strategies[i] = s
    s.Set_ID(i)

In [7]:
number_of_games = 10
total_games_explicit = True
max_action_memory = -1

gm = Game_Master(prison, strategies=strategies, duel_size=2, max_action_memory=max_action_memory)
duel_matrix, rewards = gm.Tournament(number_of_games, Game_Master.Game_Type.All_Vs_All, total_games_explicit=total_games_explicit)
rewards.Sort_Total_Rewards()

In [8]:
print(duel_matrix.Get_Action_History((0, 1)).Get_Action_History())
print(Get_Index_By_Name(strategies, "Periodic"))

{0: [<Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>], 1: [<Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>]}
-1


In [9]:
total_rewards = rewards.Get_Total_Rewards()
total_rewards_per_name = {str(strategies[i]):total_rewards[i] for i in total_rewards.keys()}

average_rewards_per_match = {k: (float(v)/number_of_strategies) for k, v in total_rewards_per_name.items()}
average_rewards_per_round = {k: (float(v)/(number_of_strategies*number_of_games)) for k, v in total_rewards_per_name.items()}

duel_rewards = rewards.Get_All_Duel_Rewards()
winner = next(iter(total_rewards))

print(total_rewards)

{2: 202, 1: 192, 5: 156, 6: 154, 7: 153, 0: 152, 3: 132, 4: 123}


In [10]:
def Sort_Based_On_Total_Rewards(total_rewards, data):
    sorted_data = dict(sorted(
        data.items(),
        key=lambda kv: total_rewards.get(kv[0], 0),
        reverse=True
    ))
    return sorted_data

## Wyniki

In [11]:
average_reward_per_round_df = pd.DataFrame.from_dict(average_rewards_per_round, orient="index", columns=["Average Reward"])
average_reward_per_round_df.index.name = "Strategy Name"
average_reward_per_round_df = average_reward_per_round_df.round(3)
average_reward_per_round_df

,Average Reward
Strategy Name,
(2):Always_Betray,2.525
(1):Random_Strategy (p_coop=0.1),2.400
(5):Patient_Unforgiving (patience=1),1.950
(6):Copycat (1st:Cooperate),1.925
(7):Periodic (period=1),1.912
(0):Random_Strategy (p_coop=0.5),1.900
(3):Always_Cooperate,1.650
(4):Always_Cooperate,1.538


## Nemesis

In [12]:
from Base_Modules.Nemesis import Nemesis_Best_Enemy_Score, Nemesis_Worst_Score, Nemesis_Largest_Difference

criterion = Nemesis_Worst_Score

nemesis = criterion.Get_Nemesis(duel_rewards=duel_rewards)
nemesis = Sort_Based_On_Total_Rewards(total_rewards=total_rewards, data=nemesis)
nemesis_per_name = criterion.Translate_Nemesis_To_Strategy_Names(strategies=strategies, nemesis=nemesis)

nemesis_df = pd.DataFrame(
    [(k, v[0]) for k, v in nemesis_per_name.items()],
    columns=["Strategy", "Its nemesis"]
)

display(nemesis_df)

,Strategy,Its nemesis
0,(2):Always_Betray,(1):Random_Strategy (p_coop=0.1)
1,(1):Random_Strategy (p_coop=0.1),(2):Always_Betray
2,(5):Patient_Unforgiving (patience=1),(1):Random_Strategy (p_coop=0.1)
3,(6):Copycat (1st:Cooperate),(1):Random_Strategy (p_coop=0.1)
4,(7):Periodic (period=1),(1):Random_Strategy (p_coop=0.1)
5,(0):Random_Strategy (p_coop=0.5),(2):Always_Betray
6,(3):Always_Cooperate,(2):Always_Betray
7,(4):Always_Cooperate,(2):Always_Betray


In [22]:
from Base_Modules.Nemesis import Friend_Best_Total_Score

criterion = Friend_Best_Total_Score

friends = criterion.Get_Nemesis(duel_rewards=duel_rewards)
friends = Sort_Based_On_Total_Rewards(total_rewards=total_rewards, data=friends)
friends_per_name = criterion.Translate_Nemesis_To_Strategy_Names(strategies=strategies, nemesis=friends)

print(friends)

largets_shared_victory = 0

# print(friends.values())

for (_, rewards) in friends.values():
    largets_shared_victory = max(largets_shared_victory, sum(rewards.values()))

# largets_shared_victory = sum(list(friends.values())[0][1].values())
print(largets_shared_victory)

friends_df = pd.DataFrame(
    [(k, v[0]) for k, v in friends_per_name.items()],
    columns=["Strategy", "Its nemesis"]
)

display(friends_df)

{2: (3, {2: 50, 3: 0}), 1: (4, {1: 46, 4: 6}), 5: (3, {3: 30, 5: 30}), 6: (3, {3: 30, 6: 30}), 7: (3, {3: 15, 7: 40}), 0: (3, {0: 34, 3: 24}), 3: (4, {3: 30, 4: 30}), 4: (3, {3: 30, 4: 30})}
60


,Strategy,Its nemesis
0,(2):Always_Betray,(3):Always_Cooperate
1,(1):Random_Strategy (p_coop=0.1),(4):Always_Cooperate
2,(5):Patient_Unforgiving (patience=1),(3):Always_Cooperate
3,(6):Copycat (1st:Cooperate),(3):Always_Cooperate
4,(7):Periodic (period=1),(3):Always_Cooperate
5,(0):Random_Strategy (p_coop=0.5),(3):Always_Cooperate
6,(3):Always_Cooperate,(4):Always_Cooperate
7,(4):Always_Cooperate,(3):Always_Cooperate


In [ ]:
# Per match

average_reward_per_match_df = pd.DataFrame.from_dict(average_rewards_per_match, orient="index", columns=["Average Reward"])
average_reward_per_match_df.index.name = "Strategy Name"
average_reward_per_match_df = average_reward_per_match_df.round(3)
# average_reward_per_match_df

In [ ]:
strat_names = [str(s) for s in strategies.values()]

score_matrix = pd.DataFrame(index=strat_names, columns=strat_names, dtype=object)

for (s1, s2), scores in duel_rewards.items():
    score_matrix.loc[str(strategies[s2]), str(strategies[s1])] = (scores[s2], scores[s1])
    score_matrix.loc[str(strategies[s1]), str(strategies[s2])] = (scores[s1], scores[s2])

for s in strat_names:
    score_matrix.loc[s, s] = (0, 0)

display(score_matrix)

,(0):Random_Strategy (p_coop=0.5),(1):Random_Strategy (p_coop=0.1),(2):Always_Betray,(3):Always_Cooperate,(4):Always_Cooperate,(5):Patient_Unforgiving (patience=1),(6):Copycat (1st:Cooperate),(7):Periodic (period=1)
(0):Random_Strategy (p_coop=0.5),"(0, 0)","(8, 28)","(2, 42)","(44, 9)","(38, 18)","(11, 31)","(30, 25)","(27, 12)"
(1):Random_Strategy (p_coop=0.1),"(28, 8)","(0, 0)","(8, 18)","(44, 9)","(50, 0)","(14, 9)","(14, 9)","(30, 5)"
(2):Always_Betray,"(42, 2)","(18, 8)","(0, 0)","(50, 0)","(50, 0)","(14, 9)","(14, 9)","(30, 5)"
(3):Always_Cooperate,"(9, 44)","(9, 44)","(0, 50)","(0, 0)","(30, 30)","(30, 30)","(30, 30)","(15, 40)"
(4):Always_Cooperate,"(18, 38)","(0, 50)","(0, 50)","(30, 30)","(0, 0)","(30, 30)","(30, 30)","(15, 40)"
(5):Patient_Unforgiving (patience=1),"(31, 11)","(9, 14)","(9, 14)","(30, 30)","(30, 30)","(0, 0)","(30, 30)","(27, 12)"
(6):Copycat (1st:Cooperate),"(25, 30)","(9, 14)","(9, 14)","(30, 30)","(30, 30)","(30, 30)","(0, 0)","(23, 28)"
(7):Periodic (period=1),"(12, 27)","(5, 30)","(5, 30)","(40, 15)","(40, 15)","(12, 27)","(28, 23)","(0, 0)"


In [ ]:
sum_matrix = pd.DataFrame(index=strat_names, columns=strat_names, dtype=object)

for (s1, s2), scores in duel_rewards.items():
    sum_matrix.loc[str(strategies[s2]), str(strategies[s1])] = (scores[s2] + scores[s1])
    sum_matrix.loc[str(strategies[s1]), str(strategies[s2])] = (scores[s2] + scores[s1])

for s in strat_names:
    sum_matrix.loc[s, s] = 0

display(sum_matrix)

,(0):Random_Strategy (p_coop=0.5),(1):Random_Strategy (p_coop=0.1),(2):Always_Betray,(3):Always_Cooperate,(4):Always_Cooperate,(5):Patient_Unforgiving (patience=1),(6):Copycat (1st:Cooperate),(7):Periodic (period=1)
(0):Random_Strategy (p_coop=0.5),0,36,44,53,56,42,55,39
(1):Random_Strategy (p_coop=0.1),36,0,26,53,50,23,23,35
(2):Always_Betray,44,26,0,50,50,23,23,35
(3):Always_Cooperate,53,53,50,0,60,60,60,55
(4):Always_Cooperate,56,50,50,60,0,60,60,55
(5):Patient_Unforgiving (patience=1),42,23,23,60,60,0,60,39
(6):Copycat (1st:Cooperate),55,23,23,60,60,60,0,51
(7):Periodic (period=1),39,35,35,55,55,39,51,0


In [ ]:
victory_matrix = score_matrix.apply(lambda col: col.map(lambda x: int(x[0] > x[1])))
for s in strat_names:
    victory_matrix.loc[s, s] = float("NaN")

In [ ]:
def color_cell(x):
    if not isinstance(x, tuple):
        return ""
    if x[0] == 0 and x[1] == 0:
        return "background-color: black"
    elif x[0] > x[1]:
        return "background-color: green"
    elif x[0] < x[1]:
        return "background-color: darkred"
    else:
        return "background-color: gray"

styled_score_matrix = (
    score_matrix.style
    .map(color_cell)
    .set_properties(**{
        "border": "1px solid black"
    })
    .set_table_styles([
        {"selector": "th", "props": [("border", "2px solid black")]},
        {"selector": "td", "props": [("border", "2px solid black")]}
    ])
)

display(styled_score_matrix)

,(0):Random_Strategy (p_coop=0.5),(1):Random_Strategy (p_coop=0.1),(2):Always_Betray,(3):Always_Cooperate,(4):Always_Cooperate,(5):Patient_Unforgiving (patience=1),(6):Copycat (1st:Cooperate),(7):Periodic (period=1)
(0):Random_Strategy (p_coop=0.5),"(0, 0)","(8, 28)","(2, 42)","(44, 9)","(38, 18)","(11, 31)","(30, 25)","(27, 12)"
(1):Random_Strategy (p_coop=0.1),"(28, 8)","(0, 0)","(8, 18)","(44, 9)","(50, 0)","(14, 9)","(14, 9)","(30, 5)"
(2):Always_Betray,"(42, 2)","(18, 8)","(0, 0)","(50, 0)","(50, 0)","(14, 9)","(14, 9)","(30, 5)"
(3):Always_Cooperate,"(9, 44)","(9, 44)","(0, 50)","(0, 0)","(30, 30)","(30, 30)","(30, 30)","(15, 40)"
(4):Always_Cooperate,"(18, 38)","(0, 50)","(0, 50)","(30, 30)","(0, 0)","(30, 30)","(30, 30)","(15, 40)"
(5):Patient_Unforgiving (patience=1),"(31, 11)","(9, 14)","(9, 14)","(30, 30)","(30, 30)","(0, 0)","(30, 30)","(27, 12)"
(6):Copycat (1st:Cooperate),"(25, 30)","(9, 14)","(9, 14)","(30, 30)","(30, 30)","(30, 30)","(0, 0)","(23, 28)"
(7):Periodic (period=1),"(12, 27)","(5, 30)","(5, 30)","(40, 15)","(40, 15)","(12, 27)","(28, 23)","(0, 0)"


In [ ]:
import webbrowser
store_styled_matrix_in_html = False
if store_styled_matrix_in_html:
    styled_score_matrix.to_html("styled_score_matrix.html")
    webbrowser.open_new_tab("styled_score_matrix.html")